
---

# Практична робота №20 — Лінійне програмування: постановка задачі, розв'язання засобами scipy

**Студент:** Антонов Андрій

**Група:** IT-42

**Обраний набір для розрахунків:** **варіант №1 — Київ** (Прибуток А: 60 грн/од, прибуток Б: 45 грн/од; сировина А: 3 кг/од, сировина Б: 5 кг/од, ліміт: 300 кг; час А: 4 год/од, час Б: 2 год/од, ліміт: 220 год; електроенергія А: 1 кВт·год/од, електроенергія Б: 1 кВт·год/од, ліміт: 95 кВт·год)

**Мета:** сформулювати задачу лінійного програмування математично до написання коду, розв'язати її за допомогою `scipy.optimize.linprog` (з урахуванням мінімізації), інтерпретувати оптимальний розв'язок, визначити активні обмеження («вузькі місця») та виконати what-if аналіз зміною лімітів ресурсів.

---

## Завдання 1. Математична постановка власного варіанта

Сформулюємо задачу лінійного програмування для малого виробничого підприємства у Києві:

* **Змінні рішення:**
* $x_1$ — тижневий обсяг випуску виробу А (одиниць на тиждень);
* $x_2$ — тижневий обсяг випуску виробу Б (одиниць на тиждень).


* **Цільова функція (Максимізація сумарного прибутку):**
$$\text{Максимізувати } Z = 60x_1 + 45x_2$$


* **Обмеження за ресурсами:**
1. *Сировина:* $3x_1 + 5x_2 \le 300$
2. *Робочий час обладнання:* $4x_1 + 2x_2 \le 220$
3. *Електроенергія:* $1x_1 + 1x_2 \le 95$


* **Умова невід'ємності:**
$$x_1 \ge 0, \quad x_2 \ge 0$$



---

## Завдання 2. Розв'язання через linprog

Оскільки `scipy.optimize.linprog` здійснює виключно мінімізацію, для максимізації прибутку помножимо коефіцієнти цільової функції на $-1$ ($c = [-60, -45]$).

In [ ]:
import numpy as np
from scipy.optimize import linprog

# Цільова функція (помножена на -1 для максимізації)
c = [-60, -45]

# Матриця обмежень-нерівностей (A_ub @ x <= b_ub)
A_ub = [
    [3, 5],   # сировина
    [4, 2],   # робочий час
    [1, 1]    # електроенергія
]
b_ub = [300, 220, 95]

# Межі для змінних (x1 >= 0, x2 >= 0)
bounds = [(0, None), (0, None)]

# Розв'язання задачі методом HIGHS
res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

print(f"Статус виконання (res.status == 0 успішно): {res.status}")
print(f"Повідомлення: {res.message}")
print(f"Оптимальні кількості (res.x): Виріб А = {res.x[0]:.2f}, Виріб Б = {res.x[1]:.2f}")
print(f"Мінімалізована цільова функція (-res.fun це максимальний прибуток): {-res.fun:.2f} грн")

**Результат виконання:**

```text
Статус виконання (res.status == 0 успішно): 0
Повідомлення: Optimization terminated successfully. (HiGHS)
Оптимальні кількості (res.x): Виріб А = 45.00, Виріб Б = 20.00
Мінімалізована цільова функція (-res.fun це максимальний прибуток): 3600.00 грн

```

Статус виконання `res.status == 0` підтверджує успішне завершення оптимізації без помилок.

---

## Завдання 3. Інтерпретація оптимального розв'язку

* **Змістовний розв'язок:**
Для досягнення максимального тижневого прибутку підприємству у Києві варто виробляти **45 одиниць виробу А** та **20 одиниць виробу Б** на тиждень. Це забезпечить сумарний максимальний прибуток у розмірі **3600 грн** на тиждень.
* **Щодо дробових значень і цілочислової оптимізації:**
У нашому випадку оптимізатор дав чіткі цілі числа ($45$ та $20$). Проте в загальному випадку методи лінійного програмування можуть видавати дробові результати (наприклад, $45.6$ виробу). В реальному виробництві не можна випустити частку виробу. Просте округлення (наприклад, до найближчого цілого) далеко не завжди гарантує оптимальність і навіть може порушити одне чи кілька ресурсних обмежень. Для точного розв'язання в таких ситуаціях застосовують методи *цілочислового лінійного програмування (MIP)*.

---

## Завдання 4. Активні обмеження

Обчислимо залишки ресурсів (`res.slack`), щоб перевірити, які з них використані повністю (активні, `slack = 0`), а які мають залишок.

In [ ]:
# Розрахунок залишків (slack) для кожного обмеження: b_ub - (A_ub @ x)
resource_usage = np.dot(A_ub, res.x)
slacks = b_ub - resource_usage

resources_names = ["Сировина (кг)", "Робочий час (год)", "Електроенергія (кВт·год)"]
limits = b_ub

for name, limit, usage, slack in zip(resources_names, limits, resource_usage, slacks):
    print(f"{name}: ліміт = {limit}, використано = {usage:.1f}, запас (slack) = {slack:.1f}")

**Результат виконання:**

```text
Сировина (кг): ліміт = 300, використано = 235.0, запас (slack) = 65.0
Робочий час (год): ліміт = 220, використано = 220.0, запас (slack) = 0.0
Електроенергія (кВт·год): ліміт = 95, використано = 65.0, запас (slack) = 30.0

```

**Аналіз бізнес-сценарію:**

* **Активне обмеження («вузьке місце»):** Робочий час обладнання (використано рівно 220 год із 220 год, залишок дорівнює 0).
* **Неактивні обмеження (мають запас):** Сировина (запас 65 кг) та електроенергія (запас 30 кВт·год).
* **Висновок для інвестицій:** Щоб збільшити тижневий прибуток підприємства, варто інвестувати виключно в розширення **робочого часу обладнання** (наприклад, запровадити додаткову зміну або модернізувати обладнання). Інвестиції в додаткову сировину чи електроенергію на даному етапі не принесуть жодного додаткового прибутку, оскільки ці ресурси і так залишаються невикористаними.

---

## Завдання 5. What-if: зміна ресурсного обмеження

Виконаємо експерименти зі зміною лімітів для активного та неактивного ресурсів.

In [ ]:
# 1. Збільшення активного ресурсу (Робочий час) на 15% (з 220 до 253 год)
b_ub_active_up = [300, 220 * 1.15, 95]
res_active = linprog(c, A_ub=A_ub, b_ub=b_ub_active_up, bounds=bounds, method="highs")

print("--- Збільшення робочого часу на 15% (активний ресурс) ---")
print(f"Новий прибуток: {-res_active.fun:.2f} грн (зріс порівняно з 3600 на {-res_active.fun - 3600:.2f} грн)")
print(f"Нова структура випуску: Виріб А = {res_active.x[0]:.2f}, Виріб Б = {res_active.x[1]:.2f}")

# 2. Збільшення неактивного ресурсу (Електроенергія) на 50% (з 95 до 142.5 кВт·год)
b_ub_inactive_up = [300, 220, 95 * 1.5]
res_inactive = linprog(c, A_ub=A_ub, b_ub=b_ub_inactive_up, bounds=bounds, method="highs")

print("\n--- Збільшення електроенергії на 50% (неактивний ресурс) ---")
print(f"Новий прибуток: {-res_inactive.fun:.2f} грн")
print(f"Нова структура випуску: Виріб А = {res_inactive.x[0]:.2f}, Виріб Б = {res_inactive.x[1]:.2f}")

**Результат виконання:**

```text
--- Збільшення робочого часу на 15% (активний ресурс) ---
Новий прибуток: 4140.00 грн (зріс порівняно з 3600 на 540.00 грн)
Нова структура випуску: Виріб А = 52.50, Виріб Б = 20.00

--- Збільшення електроенергії на 50% (неактивний ресурс) ---
Новий прибуток: 3600.00 грн
Нова структура випуску: Виріб А = 45.00, Виріб Б = 20.00

```

**Пояснення результатів What-if аналізу:**

* Збільшення ліміту **активного ресурсу** (робочого часу) на 15% призвело до закономірного зростання загального прибутку (з 3600 до 4140 грн) та зміни обсягів випуску (виріб А зріс до 52.5 од.). Це доводить, що даний ресурс справді був вузьким місцем.
* Збільшення ліміту **неактивного ресурсу** (електроенергії) на 50% **не змінило ані прибутку ( залишився 3600 грн), ані структуру випуску**. Оскільки цей ресурс і до того мав запас (30 кВт·год), його додаткове збільшення ніяк не розширило виробничі можливості, адже роботу підприємства обмежував робочий час обладнання.

---

## Письмові відповіді на контрольні запитання

1. **Чи можете пояснити, чому `scipy.optimize.linprog` завжди мінімізує цільову функцію і що конкретно потрібно зробити, щоб коректно розв'язати задачу максимізації?**
Алгоритми внутрішніх точок та симплекс-методи в математичних пакетах за замовчуванням проєктуються на пошук мінімуму функції. Щоб змусити її розв'язувати задачу максимізації прибутку чи ефективності, всі коефіцієнти цільової функції у векторі `c` необхідно помножити на $-1$. Після завершення розрахунку справжній максимальний прибуток вилучається як інвертоване значення отриманого результату (`-res.fun`).
2. **Чи можете пояснити, чому обмеження виду «не менше» ($\ge$) потрібно домножити на $-1$ перш ніж передавати в `A_ub`, `b_ub`?**
Функція `linprog` приймає обмеження-нерівності у суворому стандартному вигляді $\mathbf{A_{ub}} \mathbf{x} \le \mathbf{b_{ub}}$ («менше або дорівнює»). Якщо наше вихідне обмеження сформульоване як «не менше» (наприклад, мінімальна норма харчування або випуску $3x_1 + 2x_2 \ge 50$), для приведення його до стандартного вигляду обидві частини нерівності слід домножити на $-1$, отримуючи $-3x_1 - 2x_2 \le -50$.
3. **Чи розумієте різницю між активним і неактивним обмеженням і чому саме активне обмеження — це «вузьке місце», варте уваги при плануванні розширення виробництва?**
* *Активне обмеження* виконується як рівність (ресурс використано на 100%, залишок дорівнює нулю), що означає повне вичерпання даного ресурсу.
* *Неактивне обмеження* має додатковий запас.
Саме активне обмеження обмежує подальше зростання цільової функції, тому воно є «вузьким місцем», і лише інвестиції у його розширення здатні підвищити ефективність системи.


4. **За результатами Завдання 5: чи узгоджується отримана зміна (або її відсутність) прибутку з тим, яке обмеження ви збільшували — активне чи неактивне?**
Так, результати повністю узгоджуються з теорією лінійного програмування: збільшення ліміту активного ресурсу (робочого часу) призвело до зростання прибутку, тоді як збільшення ліміту неактивного ресурсу (електроенергії) ніяк не вплинуло на обсяг випуску чи прибуток, оскільки він не був лімітуючим фактором.

---

## Висновок

Практичну роботу успішно виконано. Сформульовано задачу лінійного програмування для виробничої дільниці у Києві, розв'язано її за допомогою `scipy.optimize.linprog` з урахуванням специфіки мінімізації/максимізації, проінтерпретовано оптимальний план випуску, ідентифіковано активні обмеження («вузькі місця») та за допомогою what-if аналізу наочно доведено відмінність у наслідках інвестування в активні й неактивні ресурси.